# Assignment 6: RAG - Machine Learning Knowledge Assistant

>### Embedding Model Used:
>##### I used SentenceTransformer all-MiniLM-L6-v2. It converts each text chunk into a numerical vector so that semantic similarity search can be performed.

>### Vector Database Used:
>##### I used FAISS. FAISS stores all chunk embeddings and retrieves the most relevant chunks for a user query using vector similarity search.

>### Prompt Design:
>##### The prompt instructs the LLM to answer only from the retrieved context. If the answer is not available in the retrieved content, it must say that the answer was not found. This reduces hallucination and keeps the answer grounded.

>### RAG Flow:
>##### PDF → Text Extraction → Chunking → Embeddings → FAISS Vector Store → Similarity Search → Retrieved Context → LLM Answer → Final Response with Page Sources.

### 1. Install required libraries

In [8]:
!pip install pypdf sentence-transformers faiss-cpu openai tiktoken -q


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:

import os
import faiss
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from openai import OpenAI


# CONFIG 
PDF_PATH = r"C:\Users\296845\Downloads\Gen_AI_Assignment_6\intro-to-ml.pdf"
os.environ["GROQ_API_KEY"] = "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"


# LOAD PDF
def load_pdf(pdf_path):
    if not os.path.exists(pdf_path):
        print("File not found. Check your path.")
        return []

    reader = PdfReader(pdf_path)
    pages_text = []

    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():
            pages_text.append({
                "page": page_num + 1,
                "text": text
            })

    return pages_text


pages = load_pdf(PDF_PATH)
print("Pages Loaded:", len(pages))


#  CHUNKING 
def chunk_text(pages, chunk_size=1200, overlap=200):
    chunks = []

    for page in pages:
        text = page["text"].replace("\n", " ").strip()

        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]

            if len(chunk.strip()) > 100:
                chunks.append({
                    "text": chunk,
                    "page": page["page"]
                })

            start = end - overlap

    return chunks


chunks = chunk_text(pages)
print("Chunks Created:", len(chunks))


#  EMBEDDINGS 
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embeddings Shape:", embeddings.shape)


#  VECTOR DB (FAISS) 
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS Index Ready. Total vectors:", index.ntotal)


# RETRIEVAL 
def retrieve_chunks(query, top_k=5):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            "rank": i + 1,
            "page": chunks[idx]["page"],
            "text": chunks[idx]["text"],
            "distance": distances[0][i]
        })

    return results


# LLM SETUP (GROQ)
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


#  ANSWER GENERATION 
def generate_answer(query, retrieved_chunks):

    context = ""
    for item in retrieved_chunks:
        context += f"\n[Page {item['page']}]\n{item['text']}\n"

    prompt = f"""
You are a Machine Learning Knowledge Assistant.

Answer ONLY using the context below.

Rules:
- Do NOT use outside knowledge
- If answer not found, say: "I could not find this in the provided book"
- Keep answer simple and clear
- Mention page numbers

Context:
{context}

Question:
{query}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": "You are a helpful ML assistant"},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


#  RAG PIPELINE
def rag_pipeline(query, top_k=5):

    retrieved = retrieve_chunks(query, top_k)
    answer = generate_answer(query, retrieved)

    print("\n Question:")
    print(query)

    print("\n Answer:")
    print(answer)

    print("\n Sources:")
    for item in retrieved:
        print(f"Rank {item['rank']} | Page {item['page']} | Distance {item['distance']:.4f}")


#  TEST 
rag_pipeline("What is supervised learning?")
rag_pipeline("Explain overfitting and underfitting")
rag_pipeline("What is k-nearest neighbors?")


#  CHAT LOOP 
while True:
    q = input("\nAsk question (type 'exit'): ")

    if q.lower() == "exit":
        print("Exiting RAG Assistant")
        break

    rag_pipeline(q)

Pages Loaded: 387
Chunks Created: 850


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Embeddings Shape: (850, 384)
FAISS Index Ready. Total vectors: 850

 Question:
What is supervised learning?

 Answer:
Supervised learning is one of the most commonly used and successful types of machine learning. It is used whenever we want to predict a certain outcome from a given input, and we have examples of input/output pairs. We build a machine learning model from these input/output pairs, which comprise our training set. Our goal is to make accurate predictions for new, never-before-seen data. (Page 39)

 Sources:
Rank 1 | Page 39 | Distance 0.8316
Rank 2 | Page 114 | Distance 0.8846
Rank 3 | Page 88 | Distance 0.9090
Rank 4 | Page 6 | Distance 0.9424
Rank 5 | Page 16 | Distance 0.9826

 Question:
Explain overfitting and underfitting

 Answer:
Overfitting and underfitting are two common issues that can occur when training a machine learning model.

Overfitting (Page 88) occurs when a model is too complex and focuses too much on the training data, resulting in poor performance on

...


 Question:
what is Machine Learning 

 Answer:
Machine Learning is about extracting knowledge from data. It is a research field at the intersection of statistics, artificial intelligence, and computer science, also known as predictive analytics or statistical learning. (Page 15)

 Sources:
Rank 1 | Page 15 | Distance 0.6249
Rank 2 | Page 15 | Distance 0.6806
Rank 3 | Page 9 | Distance 0.8160
Rank 4 | Page 289 | Distance 0.8174
Rank 5 | Page 380 | Distance 0.8226



Ask question (type 'exit'):  How Machine Learning is different from Artificial intelligence 



 Question:
How Machine Learning is different from Artificial intelligence 

 Answer:
I could not find this in the provided book

 Sources:
Rank 1 | Page 15 | Distance 0.8654
Rank 2 | Page 16 | Distance 0.8877
Rank 3 | Page 5 | Distance 0.8943
Rank 4 | Page 131 | Distance 0.8990
Rank 5 | Page 15 | Distance 0.9717



Ask question (type 'exit'):  exit


Exiting RAG Assistant
